# Proyecto LSP — Exportar el modelo a TensorFlow.js (para la demo web)

Convierte `modelo_pucp305_final.keras` → **TensorFlow.js GraphModel** (`model.json` + pesos)
para que la demo HTML (`web/lsp_demo.html`) corra el modelo en el navegador, sin Python.

Genera un `web_model.zip` que descargas y descomprimes en `web/web_model/` (junto al HTML).

> **Por qué GraphModel y no LayersModel:** convertir el BiLSTM como *LayersModel*
> (`save_keras_model`) reconstruye las capas y falla en el navegador por el nombrado de
> pesos del Bidirectional de Keras 3 (*"no target variable: forward_lstm/..."*). El
> **GraphModel** exporta un grafo congelado que TF.js solo ejecuta → evita ese problema.
> El HTML lo carga con `tf.loadGraphModel`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# tensorflowjs instala sus propias deps, INCLUIDA jax (su converter la importa).
# OJO: a diferencia de los notebooks de inferencia (04/05/06-setup), aquí NO se
# desinstala jax — hacerlo rompe `import tensorflowjs` (ModuleNotFoundError: jax).
!pip install -q tensorflowjs

In [ ]:
import shutil, subprocess
from pathlib import Path
import tensorflow as tf

M  = Path('/content/drive/MyDrive/PUCP305_models')
SM = Path('/content/saved_model')           # SavedModel intermedio
OUT = Path('/content/web_model'); OUT.mkdir(exist_ok=True)

model = tf.keras.models.load_model(M / 'modelo_pucp305_final.keras', compile=False)
print('Modelo cargado:', model.input_shape, '->', model.output_shape)

# 1) Keras 3 -> SavedModel (grafo con firma serving_default)
if SM.exists():
    shutil.rmtree(SM)
model.export(str(SM))

# 2) SavedModel -> TF.js GRAPH model (grafo congelado).
#    A diferencia de loadLayersModel, el GraphModel solo EJECUTA el grafo, asi que
#    NO reconstruye las capas y evita el bug de nombres de pesos del BiLSTM.
cmd = ['tensorflowjs_converter', '--input_format=tf_saved_model',
       '--output_format=tfjs_graph_model',
       '--signature_name=serving_default', '--saved_model_tags=serve',
       str(SM), str(OUT)]
print('\n$', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout); print(r.stderr)
r.check_returncode()
print('Conversion a GraphModel OK.')

# clases junto al modelo (las usa el HTML)
shutil.copy(M / 'classes_pucp305.json', OUT / 'classes_pucp305.json')
print('\nArchivos generados:')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:30} {p.stat().st_size/1024:8.1f} KB')

In [ ]:
# Empaquetar y descargar
shutil.make_archive('/content/web_model', 'zip', OUT)
from google.colab import files
files.download('/content/web_model.zip')
print('Descomprime web_model.zip dentro de la carpeta web/ del repo (junto a lsp_demo.html).')